# Electrode statistics of both tasks in one table (supplementary overview)

This notebook only **collects** results; it runs no test and reads no run data. It reads the statistics caches written by
* `quantifying_attr_on_arousal.ipynb` → `results/apa_mwu/arousal_stats.json` (arousal prediction, 15 runs, 60 electrodes)
* `quantifying_attr_on_physiomi.ipynb` → `results/apa_mwu/physiomi_stats.json` (PhysioMI left/right hand, 15 runs, 64 electrodes)

and lists each task's electrodes in its own panel of one table (no electrode-by-electrode pairing across tasks):
* **electrode vs. background** (one-sided Mann–Whitney U): median (IQR) relative relevance of the electrode, *z*, *p*,
  FDR-adjusted *p* and the effect size *r*.

The LaTeX tables concern only this question, whether an electrode's relevance is significantly higher than the
background (non-random). The Spearman correlations with test balanced accuracy are reported in the source notebooks
and kept in the CSV overview here, not in the tables.

FDR correction was done within each task (across that task's electrodes), exactly as in the source notebooks.

**Rerun the two source notebooks first** if their data or code changed; section 1 prints when and from which commit
each cache was written.

**Output** (`results/apa_mwu/`): `electrode_summary_both_tasks_significant.tex` (per task, the electrodes FDR-significant
against the background), `electrode_summary_both_tasks_all.tex` (all electrodes, `longtable`) and
`electrode_summary_both_tasks.csv`.

## Statistical test and reporting helpers

All group comparisons in this notebook use the **Mann–Whitney U test** (`scipy.stats.mannwhitneyu`) with
**normal approximation, tie correction and no continuity correction**.
Without ties this yields exactly the z and p of `scipy.stats.ranksums`, which earlier versions of the analysis used.

Every test reports, in APA style:
* each group characterized by *n*, median (Mdn) and interquartile range (IQR = Q1–Q3),
* *U* (statistic of the first-named group: number of pairs in which its value is larger, ties counting ½),
* *z* (tie-corrected; *z* > 0 means the first-named group tends to be larger),
* *p* (and the FDR-adjusted *p* where several tests are run),
* the effect size *r* = *z* / √*N*, and *A* = *U* / (*n*₁*n*₂), the probability that a random value of the first group exceeds one of the second.

The cell also defines the booktabs (LaTeX) table builder and `write_output`, which saves the `.tex` tables and a
`.json` statistics cache (inputs, results, code and library versions, data locations) to `results/apa_mwu/`.
This cell is identical in the four notebooks that report these statistics (listed at the top of the cell).

In [1]:
# ---- Mann–Whitney U test with APA reporting, booktabs tables and a statistics cache ----
# This cell is identical in quantifying_attr_on_arousal.ipynb, quantifying_attr_on_physiomi.ipynb,
# clever_hans_lateral_holdout_mwu.ipynb and electrode_statistics_summary.ipynb.
import datetime
import json
import platform
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy.stats import mannwhitneyu, norm

# Repository root (folder holding pyproject.toml), independent of where the kernel was started.
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
STATS_DIR = PROJECT_ROOT / "results" / "apa_mwu"  # .tex tables and .json statistics caches are written here

# Normal approximation with tie correction, without continuity correction.
# Without ties this gives exactly the z and p of scipy.stats.ranksums (checked below where ranksums was used before).
MWU_SETTINGS = {"method": "asymptotic", "use_continuity": False}
MWU_SETTINGS_TEXT = "normal approximation with tie correction, no continuity correction"


def mwu_apa(x, y, alternative, label_x, label_y):
    """Mann–Whitney U test of x vs. y, returning every quantity needed for APA reporting.

    U    statistic of x: number of pairs (x_i, y_j) with x_i > y_j, ties counting 1/2 (scipy's convention).
    z    (U - n_x n_y / 2) / sigma_U with the tie-corrected sigma_U; z > 0 means x tends to be larger than y.
    p    from z under the given alternative ("greater": x tends to be larger than y).
    r    effect size z / sqrt(n_x + n_y).
    A    U / (n_x n_y), the probability that a random x exceeds a random y (common-language effect size).
    Mdn, Q1, Q3 describe each group (IQR = Q1–Q3, linear interpolation).
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n_x, n_y = len(x), len(y)
    n = n_x + n_y
    res = mannwhitneyu(x, y, alternative=alternative, **MWU_SETTINGS)
    u = float(res.statistic)
    _, tie_sizes = np.unique(np.concatenate([x, y]), return_counts=True)
    sigma_u = np.sqrt(n_x * n_y / 12 * ((n + 1) - np.sum(tie_sizes**3 - tie_sizes) / (n * (n - 1))))
    z = (u - n_x * n_y / 2) / sigma_u
    # scipy does not return z; the p-value recomputed from our z must equal scipy's p-value
    p_from_z = {"greater": norm.sf(z), "less": norm.cdf(z), "two-sided": 2 * norm.sf(abs(z))}[alternative]
    assert np.isclose(p_from_z, res.pvalue, rtol=1e-9, atol=0), (p_from_z, res.pvalue)
    (q1_x, mdn_x, q3_x), (q1_y, mdn_y, q3_y) = np.percentile(x, [25, 50, 75]), np.percentile(y, [25, 50, 75])
    return {
        "label_x": label_x, "label_y": label_y, "alternative": alternative,
        "n_x": n_x, "mdn_x": mdn_x, "q1_x": q1_x, "q3_x": q3_x,
        "n_y": n_y, "mdn_y": mdn_y, "q1_y": q1_y, "q3_y": q3_y,
        "U": u, "z": z, "p": float(res.pvalue), "r": z / np.sqrt(n), "A": u / (n_x * n_y),
        "n_tied_values": int(np.sum(tie_sizes[tie_sizes > 1])),  # observations sharing their value with another one
    }


# ---- APA number formatting ----
def fmt_bounded(v, digits=2):
    """Quantities that cannot exceed 1 in absolute value (p, r, rho): no leading zero."""
    return f"{v:.{digits}f}".replace("0.", ".", 1)


def fmt_p(p):
    """'< .001', '> .999' or '= .023'."""
    if p < 0.001:
        return "< .001"
    if p > 0.999:
        return "> .999"
    return "= " + fmt_bounded(p, 3)


def fmt_u(u):
    return f"{u:.1f}".removesuffix(".0")  # U is an integer or ends in .5 (ties)


def describe_group(s, which, digits=3):
    """Characterization of one group, e.g. 'relevance at AF7 (n = 15, Mdn = .041, IQR = .027–.052)'.

    All compared quantities here (relative relevance, balanced accuracy) cannot exceed 1: no leading zero (APA).
    """
    f = lambda v: fmt_bounded(v, digits)
    return (f"{s['label_' + which]} (n = {s['n_' + which]}, Mdn = {f(s['mdn_' + which])}, "
            f"IQR = {f(s['q1_' + which])}–{f(s['q3_' + which])})")


def apa_mwu_text(s, p_adj=None):
    """'U = 11204, z = 4.12, p < .001, r = .14' (with 'p_FDR ...' if an adjusted p-value is given)."""
    text = f"U = {fmt_u(s['U'])}, z = {s['z']:.2f}, p {fmt_p(s['p'])}"
    if p_adj is not None:
        text += f", p_FDR {fmt_p(p_adj)}"
    return text + f", r = {fmt_bounded(s['r'])}"


def apa_mwu_sentence(s, alpha=0.05, p_adj=None, digits=3, describe_y=True):
    """Full sentence: both groups characterized, the direction, and the test statistics.

    Significance is decided on p_adj if given, otherwise on p.
    describe_y=False names the second group without n, Mdn and IQR, for a group that is characterized once
    for many tests (see describe_background).
    """
    significant = (p_adj if p_adj is not None else s["p"]) < alpha
    if s["alternative"] == "greater":
        verdict = "was significantly higher than" if significant else "was not significantly higher than"
    elif s["alternative"] == "less":
        verdict = "was significantly lower than" if significant else "was not significantly lower than"
    elif significant:
        verdict = "was significantly higher than" if s["z"] > 0 else "was significantly lower than"
    else:
        verdict = "did not differ significantly from"
    first = describe_group(s, "x", digits)
    second = describe_group(s, "y", digits) if describe_y else s["label_y"]
    return f"{first[0].upper()}{first[1:]} {verdict} {second}, {apa_mwu_text(s, p_adj)}."


def fmt_range(values, digits=3, dash="–"):
    """'.014' if all values round to the same number, else 'min–max', e.g. '.024–.025'."""
    lo, hi = fmt_bounded(min(values), digits), fmt_bounded(max(values), digits)
    return lo if lo == hi else f"{lo}{dash}{hi}"


def describe_background(results, digits=3, tex=False):
    """One phrase characterizing the background group of all electrode-vs-background tests at once.

    The background of electrode c is all other electrodes, so it hardly changes from test to test. Its n, Mdn, Q1 and
    Q3 are given as ranges over the tests (one value where all tests round to the same number); "virtually identical"
    is only stated if every range spans at most one unit of the last digit shown. The per-test values stay in the
    results (n_y, mdn_y, q1_y, q3_y).
    """
    dash = "--" if tex else "–"
    stats = ", ".join(f"{name} = {fmt_range(results[col], digits, dash)}" for name, col in [("Mdn", "mdn_y"), ("Q1", "q1_y"), ("Q3", "q3_y")])
    n_y = sorted(set(results["n_y"]))
    n_text = str(n_y[0]) if len(n_y) == 1 else f"{n_y[0]}{dash}{n_y[-1]}"
    n_text = f"$n = {n_text}$" if tex else f"n = {n_text}"
    identical = all(results[col].max() - results[col].min() <= 10 ** -digits for col in ["mdn_y", "q1_y", "q3_y"])
    verdict = "was virtually identical in all" if identical else "varied across the"
    return (f"The background group, the relative relevance of all other electrodes in all runs ({n_text} per test), "
            f"{verdict} {len(results)} tests ({stats} across tests).")


# ---- LaTeX (booktabs) output; needs \usepackage{booktabs} (and \usepackage{longtable} for long=True) ----
def tex_p(p):
    return "$< .001$" if p < 0.001 else ("$> .999$" if p > 0.999 else f"${fmt_bounded(p, 3)}$")


def tex_num(v, digits=2, bounded=False):
    return f"${fmt_bounded(v, digits) if bounded else f'{v:.{digits}f}'}$"  # math mode gives a proper minus sign


def booktabs(header, rows, colspec, caption, label, long=False):
    """booktabs table in \\small with narrower column spacing.

    header: list of cells (one header row), or list of rows, where a row is a list of cells or a raw line (e.g. a \\cmidrule).
    long=False: floating table. long=True: longtable that breaks across pages (header repeated), for full electrode lists.
    An empty table gets one row saying so.
    """
    n_cols = sum(ch in "lcr" for ch in colspec)
    rows_of_header = header if isinstance(header[0], list) else [header]
    head = "\n".join(r if isinstance(r, str) else " & ".join(r) + r" \\" for r in rows_of_header)
    body = [" & ".join(row) + r" \\" for row in rows] or [rf"\multicolumn{{{n_cols}}}{{l}}{{No electrode with $p_{{\mathrm{{FDR}}}} < .05$}} \\"]
    if long:
        return "\n".join([
            r"{\small\setlength{\tabcolsep}{4pt}",
            rf"\begin{{longtable}}{{{colspec}}}",
            rf"\caption{{{caption}}}\label{{{label}}} \\",
            r"\toprule", head, r"\midrule", r"\endfirsthead",
            rf"\multicolumn{{{n_cols}}}{{l}}{{\emph{{(continued)}}}} \\",
            r"\toprule", head, r"\midrule", r"\endhead",
            r"\bottomrule", r"\endfoot",
            *body,
            r"\end{longtable}",
            r"}",
        ])
    return "\n".join([
        r"\begin{table}[ht]",
        r"\centering",
        r"\small",
        r"\setlength{\tabcolsep}{4pt}",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        rf"\begin{{tabular}}{{{colspec}}}",
        r"\toprule",
        head,
        r"\midrule",
        *body,
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ])


# ---- Output files ----
def _git(*args):
    return subprocess.run(["git", *args], cwd=PROJECT_ROOT, capture_output=True, text=True).stdout.strip()


def provenance(data_sources):
    """Everything needed to trace the numbers back: code version, library versions, data locations, test settings."""
    return {
        "created": datetime.datetime.now().isoformat(timespec="seconds"),
        "git_commit": _git("rev-parse", "HEAD"),
        "git_branch": _git("rev-parse", "--abbrev-ref", "HEAD"),
        "git_uncommitted_changes": bool(_git("status", "--porcelain")),
        "python": platform.python_version(),
        "numpy": np.__version__,
        "scipy": scipy.__version__,
        "pandas": pd.__version__,
        "mwu_settings": {**MWU_SETTINGS, "tie_correction": True, "description": MWU_SETTINGS_TEXT},
        "data_sources": data_sources,
    }


def _jsonable(o):
    if isinstance(o, np.generic):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (pd.DataFrame, pd.Series)):
        return json.loads(o.to_json(orient="records" if isinstance(o, pd.DataFrame) else "index"))
    if isinstance(o, Path):
        return str(o)
    raise TypeError(type(o))


def write_output(filename, content):
    """Write a .tex table (str) or a .json statistics cache (dict) to STATS_DIR and print the path."""
    STATS_DIR.mkdir(parents=True, exist_ok=True)
    path = STATS_DIR / filename
    if isinstance(content, str):
        path.write_text(content + "\n")
    else:
        path.write_text(json.dumps(content, indent=1, default=_jsonable, ensure_ascii=False) + "\n")
    print(f"written: {path}")

## 1. Load the statistics caches of both tasks

Also prints, per task, the one-sentence description of the background group.

In [2]:
TASKS = {"Arousal": "arousal", "PhysioMI": "physiomi"}  # table label -> cache file stem

caches = {task: json.loads((STATS_DIR / f"{stem}_stats.json").read_text()) for task, stem in TASKS.items()}
for task, cache in caches.items():
    p = cache["provenance"]
    print(f"{task}: {cache['description']}\n"
          f"  {len(cache['inputs']['runs'])} runs, {len(cache['inputs']['channel_names'])} electrodes; "
          f"written {p['created']} from commit {p['git_commit'][:7]} (scipy {p['scipy']})")

# both tasks must have been tested with the same settings
settings = {task: cache["provenance"]["mwu_settings"] for task, cache in caches.items()}
assert len({json.dumps(s, sort_keys=True) for s in settings.values()}) == 1, settings
print(f"\nMann–Whitney U settings (both tasks): {MWU_SETTINGS_TEXT}")

background = {task: pd.DataFrame(cache["electrode_vs_background_mwu"]["results"]) for task, cache in caches.items()}
for task, results in background.items():
    print(f"{task}: {describe_background(results)}")

Arousal: Arousal prediction, frozen_arousal, from_scratch_arousal and finetuned_arousal, 5 runs each
  15 runs, 60 electrodes; written 2026-10-30T13:39:01 from commit 4296fde (scipy 1.17.1)
PhysioMI: PhysioMI left/right hand, frozen, from_scratch and finetuned LaBraM, 5 runs each
  15 runs, 64 electrodes; written 2026-10-30T13:31:13 from commit 4296fde (scipy 1.17.1)

Mann–Whitney U settings (both tasks): normal approximation with tie correction, no continuity correction
Arousal: The background group, the relative relevance of all other electrodes in all runs (n = 885 per test), was virtually identical in all 60 tests (Mdn = .014, Q1 = .006, Q3 = .024–.025 across tests).
PhysioMI: The background group, the relative relevance of all other electrodes in all runs (n = 945 per test), was virtually identical in all 64 tests (Mdn = .012–.013, Q1 = .006, Q3 = .021 across tests).


## 2. Overview of both tasks

For inspection and the CSV, one row per electrode (union of both montages), both tasks side by side. Rows are ordered by the number of tasks in which the electrode is
FDR-significant against the background, then by its smallest FDR-adjusted *p*.

In [3]:
def task_results(cache):
    mwu = pd.DataFrame(cache["electrode_vs_background_mwu"]["results"]).set_index("channel_name")
    spearman = pd.DataFrame(cache["spearman_relevance_vs_test_balanced_accuracy"]["results"]).set_index("channel_name")
    return pd.concat([
        mwu[["mdn_x", "q1_x", "q3_x", "z", "p", "p_adj", "r", "sig_fdr"]],
        spearman[["rho", "p_adj", "sig_fdr"]].rename(columns={"p_adj": "rho_p_adj", "sig_fdr": "rho_sig_fdr"}),
    ], axis=1)


combined = pd.concat({task: task_results(cache) for task, cache in caches.items()}, axis=1)  # outer join on electrodes
sig = pd.DataFrame({task: combined[(task, "sig_fdr")].astype("boolean").fillna(False) for task in caches})
combined[("both", "n_tasks_significant")] = sig.sum(axis=1)
combined[("both", "min_p_adj")] = combined[[(task, "p_adj") for task in caches]].min(axis=1)
combined = combined.sort_values([("both", "n_tasks_significant"), ("both", "min_p_adj")], ascending=[False, True])
combined.index.name = "electrode"

recorded = {task: set(combined.index[combined[(task, "z")].notna()]) for task in caches}
print(f"Electrodes: {len(combined)} in total; only in arousal: {sorted(recorded['Arousal'] - recorded['PhysioMI'])}; "
      f"only in PhysioMI: {sorted(recorded['PhysioMI'] - recorded['Arousal'])}\n")
print("FDR-significantly higher relevance than the background:")
print(f"  both tasks:    {list(sig.index[sig.all(axis=1)])}")
print(f"  arousal only:  {list(sig.index[sig['Arousal'] & ~sig['PhysioMI']])}")
print(f"  PhysioMI only: {list(sig.index[sig['PhysioMI'] & ~sig['Arousal']])}")
for task in caches:
    rho_sig = combined.index[combined[(task, "rho_sig_fdr")].astype("boolean").fillna(False)]
    print(f"FDR-significant Spearman correlation with test balanced accuracy, {task}: {list(rho_sig)}")

display(combined.round(3))

Electrodes: 66 in total; only in arousal: ['TP10', 'TP9']; only in PhysioMI: ['FCz', 'Fp1', 'Fp2', 'Fpz', 'T10', 'T9']

FDR-significantly higher relevance than the background:
  both tasks:    []
  arousal only:  ['F3', 'CP5', 'CP6', 'CP2', 'Cz', 'T8', 'FC6', 'FC2', 'C5', 'TP7', 'POz', 'P6', 'CPz', 'TP8', 'C2']
  PhysioMI only: ['F7', 'F8', 'AF7', 'AF3', 'F5', 'AF8', 'AF4', 'Fp1', 'Fpz', 'Fp2', 'T9']
FDR-significant Spearman correlation with test balanced accuracy, Arousal: []
FDR-significant Spearman correlation with test balanced accuracy, PhysioMI: ['Fp2', 'FT7']


Arousal                                                          \
            mdn_x   q1_x   q3_x      z      p p_adj      r sig_fdr    rho   
electrode                                                                   
CP2         0.037  0.033  0.043  5.011  0.000   0.0  0.167    True  0.121   
C2          0.043  0.022  0.050  4.816  0.000   0.0  0.161    True  0.596   
Cz          0.035  0.026  0.037  4.604  0.000   0.0  0.153    True -0.111   
AF7         0.009  0.004  0.013 -1.783  0.963   1.0 -0.059   False -0.057   
AF8         0.004  0.003  0.012 -2.859  0.998   1.0 -0.095   False -0.196   
...           ...    ...    ...    ...    ...   ...    ...     ...    ...   
F2          0.005  0.003  0.013 -2.873  0.998   1.0 -0.096   False -0.425   
Iz          0.007  0.005  0.008 -3.080  0.999   1.0 -0.103   False  0.054   
FCz           NaN    NaN    NaN    NaN    NaN   NaN    NaN     NaN    NaN   
TP9         0.008  0.005  0.011 -2.245  0.988   1.0 -0.075   False -0.193   
TP10        0.012  0.008  0.020 -0.556  0.711   1.0 -0.019   False -0.539   

                     ... PhysioMI                                      \
          rho_p_adj  ...        z      p  p_adj      r sig_fdr    rho   
electrode            ...                                                
CP2           0.862  ...   -0.347  0.636  0.999 -0.011   False  0.229   
C2            0.315  ...    1.438  0.075  0.301  0.046   False -0.500   
Cz            0.862  ...   -1.245  0.893  0.999 -0.040   False -0.446   
AF7           0.941  ...    4.679  0.000  0.000  0.151    True -0.507   
AF8           0.818  ...    4.591  0.000  0.000  0.148    True -0.614   
...             ...  ...      ...    ...    ...    ...     ...    ...   
F2            0.457  ...   -1.217  0.888  0.999 -0.039   False  0.325   
Iz            0.941  ...   -1.644  0.950  0.999 -0.053   False -0.046   
FCz             NaN  ...   -0.757  0.775  0.999 -0.024   False -0.068   
TP9           0.818  ...      NaN    NaN    NaN    NaN     NaN    NaN   
TP10          0.315  ...      NaN    NaN    NaN    NaN     NaN    NaN   

                                               both            
          rho_p_adj rho_sig_fdr n_tasks_significant min_p_adj  
electrode                                                      
CP2           0.733       False                   1     0.000  
C2            0.182       False                   1     0.000  
Cz            0.254       False                   1     0.000  
AF7           0.182       False                   1     0.000  
AF8           0.105       False                   1     0.000  
...             ...         ...                 ...       ...  
F2            0.524       False                   0     0.999  
Iz            0.908       False                   0     0.999  
FCz           0.896       False                   0     0.999  
TP9             NaN         NaN                   0     1.000  
TP10            NaN         NaN                   0     1.000  

[66 rows x 24 columns]

## 3. LaTeX tables (booktabs) and CSV

One panel per task, electrodes sorted by *p* within each panel.

Requires `\usepackage{booktabs}` and, for the full table, `\usepackage{longtable}`.

In [4]:
PANELS = {"Arousal": "Arousal", "PhysioMI": "PhysioMI L/R"}  # task -> panel title
N_COLS = 6


def task_rows(task, tab):
    rows = []
    for electrode, row in tab.sort_values((task, "p")).iterrows():
        f = lambda col: fmt_bounded(row[(task, col)], 3)
        rows.append([electrode, f"{f('mdn_x')} ({f('q1_x')}--{f('q3_x')})", tex_num(row[(task, "z")]),
                     tex_p(row[(task, "p")]), tex_p(row[(task, "p_adj")]), tex_num(row[(task, "r")], bounded=True)])
    return rows or [[rf"\multicolumn{{{N_COLS}}}{{l}}{{none}}"]]


def summary_table(select, which, long=False):
    rows = []
    for i, (task, title) in enumerate(PANELS.items()):
        tab = combined[combined[(task, "z")].notna()]  # electrodes recorded in this task
        panel_head = rf"\multicolumn{{{N_COLS}}}{{l}}{{\textit{{{title}}}}}"
        rows.append([(r"\midrule " if i else "") + panel_head])
        rows += task_rows(task, tab[select(tab, task)])
    n = {task: len(cache["inputs"]["runs"]) for task, cache in caches.items()}
    caption = (
        f"Electrode relevance, one panel per task: {which}. El.: electrode. Mdn (IQR: Q1--Q3): relative relevance of "
        f"the electrode across runs (arousal: $n = {n['Arousal']}$, PhysioMI: $n = {n['PhysioMI']}$ runs); $z$, $p$, "
        f"$p_{{\mathrm{{FDR}}}}$ and $r$: one-sided Mann--Whitney $U$ test of the electrode against all other electrodes "
        f"({MWU_SETTINGS_TEXT}); FDR: Benjamini--Hochberg across the electrodes of each task; $r = z / \\sqrt{{N}}$. "
        + " ".join(f"{task}: {describe_background(results, tex=True)}" for task, results in background.items())
    )
    header = ["El.", "Mdn (IQR)", "$z$", "$p$", r"$p_{\mathrm{FDR}}$", "$r$"]
    return booktabs(header, rows, "llrrrr", caption, f"tab:electrode_summary_both_tasks{'_all' if long else ''}", long=long)


significant = lambda tab, task: tab[(task, "sig_fdr")].astype(bool)
tex_significant = summary_table(significant, "per task, the electrodes with FDR-significantly higher relevance than the background")
print(tex_significant)
write_output("electrode_summary_both_tasks_significant.tex", tex_significant)
write_output("electrode_summary_both_tasks_all.tex",
             summary_table(lambda tab, task: pd.Series(True, index=tab.index), "all electrodes", long=True))

flat = combined.copy()
flat.columns = [f"{task}_{col}" for task, col in flat.columns]
flat.to_csv(STATS_DIR / "electrode_summary_both_tasks.csv")
print(f"written: {STATS_DIR / 'electrode_summary_both_tasks.csv'}")


\begin{table}[ht]
\centering
\small
\setlength{\tabcolsep}{4pt}
\caption{Electrode relevance, one panel per task: per task, the electrodes with FDR-significantly higher relevance than the background. El.: electrode. Mdn (IQR: Q1--Q3): relative relevance of the electrode across runs (arousal: $n = 15$, PhysioMI: $n = 15$ runs); $z$, $p$, $p_{\mathrm{FDR}}$ and $r$: one-sided Mann--Whitney $U$ test of the electrode against all other electrodes (normal approximation with tie correction, no continuity correction); FDR: Benjamini--Hochberg across the electrodes of each task; $r = z / \sqrt{N}$. Arousal: The background group, the relative relevance of all other electrodes in all runs ($n = 885$ per test), was virtually identical in all 60 tests (Mdn = .014, Q1 = .006, Q3 = .024--.025 across tests). PhysioMI: The background group, the relative relevance of all other electrodes in all runs ($n = 945$ per test), was virtually identical in all 64 tests (Mdn = .012--.013, Q1 = .006, Q3 = .021 acr

<>:25: SyntaxWarning: invalid escape sequence '\m'
<>:25: SyntaxWarning: invalid escape sequence '\m'
/viper/ptmp2/$USER/tmp/ipykernel_206343/424534505.py:25: SyntaxWarning: invalid escape sequence '\m'
  f"$p_{{\mathrm{{FDR}}}}$ and $r$: one-sided Mann--Whitney $U$ test of the electrode against all other electrodes "
